# 03 - Feature Engineering

Build the modeling dataset from `jira_issues_cleaned.csv`. The target remains `duration_days -> duration_category`; the data shaping removes noisy boundary records, caps project/class dominance, and balances classes so Short and Long-running do not overwhelm Standard.

## 03-01 Import feature engineering tools

Load the lightweight utilities used in this notebook. pathlib handles project-relative paths, and pandas handles feature creation, filtering, and export.

In [1]:
from pathlib import Path
import pandas as pd

## 03-02 Load cleaned source data

Read the cleaned dataset from the previous notebook, copy it for modeling work, and convert timestamp fields back into datetime values.

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "jira_issues_cleaned.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

jira_df = pd.read_csv(INPUT_PATH)
task_df = jira_df.copy()

for column in ["created", "resolutiondate"]:
    task_df[column] = pd.to_datetime(task_df[column], errors="coerce")

print(f"Cleaned source rows: {task_df.shape[0]:,}")
print(f"Cleaned source columns: {task_df.shape[1]:,}")

Cleaned source rows: 937,203
Cleaned source columns: 17


## 03-03 Create and filter duration values

Calculate duration_days from the created and resolution timestamps, then keep durations within the usable modeling range.

In [3]:
task_df["duration_days"] = (
    task_df["resolutiondate"] - task_df["created"]
).dt.total_seconds() / (60 * 60 * 24)

rows_before = len(task_df)
task_df = task_df[
    task_df["duration_days"].notna()
    & (task_df["duration_days"] >= (2 / 24))
    & (task_df["duration_days"] <= 90)
].copy()

print(f"Rows before duration filtering: {rows_before:,}")
print(f"Rows after duration filtering: {len(task_df):,}")
task_df["duration_days"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

Rows before duration filtering: 937,203


Rows after duration filtering: 578,204


count    578204.000000
mean         15.878555
std          21.139684
min           0.083333
25%           1.239546
50%           6.050874
75%          21.893226
90%          49.805259
95%          66.695716
99%          84.510098
max          89.999618
Name: duration_days, dtype: float64

## 03-04 Assign duration categories

Convert numeric durations into the three target classes used by the classifier and inspect the initial class balance.

In [4]:
def duration_category(days):
    if days <= 3:
        return "Short"
    if days <= 15:
        return "Standard"
    return "Long-running"


duration_order = ["Short", "Standard", "Long-running"]
task_df["duration_category"] = task_df["duration_days"].apply(duration_category)

class_summary = pd.DataFrame({
    "count": task_df["duration_category"].value_counts().reindex(duration_order),
    "percent": task_df["duration_category"]
        .value_counts(normalize=True)
        .reindex(duration_order)
        .mul(100)
        .round(2),
})

class_summary

,count,percent
duration_category,,
Short,213274,36.89
Standard,179831,31.10
Long-running,185099,32.01


## 03-05 Keep full duration class ranges

Keep valid examples across the full class ranges so the final balanced dataset has enough rows for model training.

In [5]:
# Keep the full valid range for each duration class.
# The earlier duration filter already removed invalid and extreme examples.
duration_window_mask = (
    (task_df["duration_category"].eq("Short") & (task_df["duration_days"] <= 3))
    | (
        task_df["duration_category"].eq("Standard")
        & task_df["duration_days"].between(3, 15, inclusive="both")
    )
    | (
        task_df["duration_category"].eq("Long-running")
        & (task_df["duration_days"] >= 15)
    )
)

rows_before = len(task_df)
task_df = task_df.loc[duration_window_mask].copy()

print(f"Rows removed outside full duration windows: {rows_before - len(task_df):,}")
print(f"Rows after duration-window check: {len(task_df):,}")

Rows removed outside full duration windows: 0
Rows after duration-window check: 578,204


## 03-06 Keep consistent project and issue groups

Retain project and issue-type combinations with enough history and a clear duration-class signal, reducing mixed groups that add label noise.

In [6]:
# Keep project/issue-type combinations where duration class has historical signal.
# This removes mixed groups that make Standard especially noisy, while retaining all classes.
group_columns = ["project_key", "issuetype_name"]
minimum_group_size = 25
minimum_category_share = 0.35

group_counts = (
    task_df
    .groupby(group_columns + ["duration_category"], observed=True)
    .size()
    .rename("category_count")
    .reset_index()
)
group_totals = (
    group_counts
    .groupby(group_columns, observed=True)["category_count"]
    .sum()
    .rename("group_count")
    .reset_index()
)
group_counts = group_counts.merge(group_totals, on=group_columns)
group_counts["category_share"] = group_counts["category_count"] / group_counts["group_count"]

consistent_groups = group_counts.loc[
    (group_counts["group_count"] >= minimum_group_size)
    & (group_counts["category_share"] >= minimum_category_share),
    group_columns + ["duration_category"],
]

rows_before = len(task_df)
task_df = task_df.merge(
    consistent_groups,
    on=group_columns + ["duration_category"],
    how="inner",
)

print(f"Rows removed from low-signal project/issue groups: {rows_before - len(task_df):,}")
print(f"Rows after consistency filtering: {len(task_df):,}")
task_df["duration_category"].value_counts().reindex(duration_order)

Rows removed from low-signal project/issue groups: 319,482
Rows after consistency filtering: 258,722


duration_category
Short           139682
Standard         37408
Long-running     81632
Name: count, dtype: int64

## 03-07 Cap project-class dominance

Limit the number of rows from each project and duration class while keeping enough examples to exceed 100,000 final rows.

In [7]:
# Prevent a few large projects from dominating the classifier.
max_rows_per_project_class = 2_500

task_df = (
    task_df
    .groupby(["project_key", "duration_category"], group_keys=False, observed=True)
    .apply(
        lambda group: group.sample(
            n=min(len(group), max_rows_per_project_class),
            random_state=42,
        )
    )
    .reset_index(drop=True)
)


print(f"Rows after project/class cap: {len(task_df):,}")
task_df["duration_category"].value_counts().reindex(duration_order)

Rows after project/class cap: 218,629


C:\Users\Omar\AppData\Local\Temp\ipykernel_13728\3248045568.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


duration_category
Short           107867
Standard         33857
Long-running     76905
Name: count, dtype: int64

## 03-08 Balance target classes

Downsample each duration category to the smallest available class size, keeping class balance without duplicating rows.

In [8]:
# Balance classes without duplicating rows. Standard is often the hardest class, so the
# final class size is anchored to the smallest available class after cleanup.
class_counts = task_df["duration_category"].value_counts()
target_class_size = int(class_counts.min())

balanced_parts = []
for category in duration_order:
    category_df = task_df.loc[task_df["duration_category"].eq(category)]
    balanced_parts.append(
        category_df.sample(n=target_class_size, random_state=42)
    )

task_df = (
    pd.concat(balanced_parts, ignore_index=True)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

balanced_summary = pd.DataFrame({
    "count": task_df["duration_category"].value_counts().reindex(duration_order),
    "percent": task_df["duration_category"]
        .value_counts(normalize=True)
        .reindex(duration_order)
        .mul(100)
        .round(2),
})

print(f"Target rows per class: {target_class_size:,}")
balanced_summary

Target rows per class: 33,857

,count,percent
duration_category,,
Short,33857,33.33
Standard,33857,33.33
Long-running,33857,33.33


## 03-09 Save final modeling dataset

Add simple calendar features, select the final model columns, and save the full and sample datasets for training and app use.

In [9]:
task_df["created_year"] = task_df["created"].dt.year
task_df["created_month"] = task_df["created"].dt.month

final_cleaned_columns = [
    "summary",
    "description",
    "priority_name",
    "issuetype_name",
    "project_key",
    "project_category_name",
    "summary_char_count",
    "summary_word_count",
    "description_char_count",
    "description_word_count",
    "has_description",
    "labels_count",
    "has_assignee",
    "votes_votes",
    "watches_watch_count",
    "created_year",
    "created_month",
    "duration_category",
]

final_cleaned_df = task_df[final_cleaned_columns].copy()

final_cleaned_path = OUTPUT_DIR / "final_cleaned.csv"
sample_path = OUTPUT_DIR / "final_cleaned_sample.csv"

final_cleaned_df.to_csv(final_cleaned_path, index=False)
final_cleaned_df.sample(n=min(100, len(final_cleaned_df)), random_state=42).to_csv(sample_path, index=False)

print(f"Saved final cleaned CSV file to: {final_cleaned_path}")
print(f"Saved sample CSV file to: {sample_path}")
print(f"Final modeling rows: {final_cleaned_df.shape[0]:,}")
print(f"Final modeling columns: {final_cleaned_df.shape[1]:,}")

Saved final cleaned CSV file to: C:\Users\Omar\Desktop\task-ml\Task-Duration-Classifier\data\processed\final_cleaned.csv
Saved sample CSV file to: C:\Users\Omar\Desktop\task-ml\Task-Duration-Classifier\data\processed\final_cleaned_sample.csv
Final modeling rows: 101,571
Final modeling columns: 18
